In [0]:
# Databricks notebook source
# Applies liquid clustering and refreshes statistics on the largest gold tables.

CATALOG = "ecom_dev"

# Materialized views are managed by the pipeline; clustering is applied to
# the underlying Delta tables that back them.
TARGETS = {
    f"{CATALOG}.gold.fct_order_items": ["order_purchase_timestamp", "seller_id"],
    f"{CATALOG}.gold.fct_payments":    ["order_date_key"],
    f"{CATALOG}.gold.fct_reviews":     ["order_date_key"],
}

for table, cluster_cols in TARGETS.items():
    cols = ", ".join(cluster_cols)
    try:
        spark.sql(f"ALTER TABLE {table} CLUSTER BY ({cols})")
        print(f"clustered  {table} by ({cols})")
    except Exception as e:
        print(f"skip cluster {table}: {type(e).__name__}")

    try:
        spark.sql(f"OPTIMIZE {table}")
        print(f"optimized  {table}")
    except Exception as e:
        print(f"skip optimize {table}: {type(e).__name__}")

    try:
        spark.sql(f"ANALYZE TABLE {table} COMPUTE STATISTICS FOR ALL COLUMNS")
        print(f"analyzed   {table}")
    except Exception as e:
        print(f"skip analyze {table}: {type(e).__name__}")

print("\nOptimization pass complete")